# Correlation between ensemble differences between different experimental parameters

### Intialize IDPET

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from idpet.visualization import plot_comparison_matrix, Visualization
from idpet.ensemble import Ensemble
from idpet.ensemble_analysis import EnsembleAnalysis
from idpet.utils import set_verbosity

# Change verbosity level to show more information when performing the analysis.
set_verbosity("INFO")

from idpet.comparison import (
    score_adaJSD, get_adaJSD_matrix,
    score_ataJSD, get_ataJSD_profile,
    score_avg_jsd,
    score_ramaJSD, get_ramaJSD_profile,
    all_vs_all_comparison,
    process_all_vs_all_output
)

import os
import re
import subprocess


/home/sosamuli/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-16 23:54:14.362250: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-16 23:54:14.372943: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-16 23:54:14.506193: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-16 23:54:14.609459: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factor

### Intialize FAIRMD databank

In [2]:
from DatabankLib.protein_functions import *
import yaml

#databankPath = "/home/sosamuli/work/NMRlipids/IDPdatabank/"  # this is the local path for the cloned Databank
#os.environ["NMLDB_ROOT_PATH"] = "/home/sosamuli/work/NMRlipids/IDPdatabank/"

databankPath = "/home/sosamuli/work/NMRlipids/IDPsimBank/"  # this is the local path for the cloned Databank
os.environ["NMLDB_ROOT_PATH"] = "/home/sosamuli/work/NMRlipids/IDPsimBank/"


# These two lines include core Databank routines and Databank API
from DatabankLib.core import *
from DatabankLib.databankLibrary import *
# This is for plotting
from DatabankLib.databankio import *
from DatabankLib.plottings import plotSimulation
#from IPython.display import display, Markdown

# This initializes the databank and stores the information of all simulations into a list.
# Each list item contains the information from README.yaml file of the given simulation.
systems = initialize_databank()

Databank initialized from the folder: /home/sosamuli/work/NMRlipids/IDPsimBank/Data/Simulations


In [3]:
import numpy as np

def compute_relaxation_differences(dict_a, dict_b):
    """
    Compute differences in T1, T2, and hetNOE between two nested dictionaries,
    and also calculate per-field RMSDs across residues.

    Parameters
    ----------
    dict_a, dict_b : dict
        Nested dictionaries of the form:
        {
            residue: {
                field: {
                    observable: {"value": float}
                }
            }
        }

    Returns
    -------
    results : dict
        {
            "differences": {
                residue: {
                    field: {
                        observable: diff (float or None)
                    }
                }
            },
            "rmsd": {
                field: {
                    observable: rmsd_value (float or None)
                }
            }
        }
        Differences are dict_a - dict_b.
    """
    differences = {}
    rmsd_data = {}  # collect differences for RMSD calculation

    residues = set(dict_a.keys()) | set(dict_b.keys())

    for res in residues:
        differences[res] = {}
        fields = set(dict_a.get(res, {}).keys()) | set(dict_b.get(res, {}).keys())
        for field in fields:
            differences[res][field] = {}
            for observable in ["T1", "T2", "hetNOE"]:
                val_a = dict_a.get(res, {}).get(field, {}).get(observable, {}).get("value")
                val_b = dict_b.get(res, {}).get(field, {}).get(observable, {}).get("value")
                if val_a is not None and val_b is not None:
                    diff = val_a - val_b
                    differences[res][field][observable] = diff
                    # store for RMSD
                    rmsd_data.setdefault(field, {}).setdefault(observable, []).append(diff)
                else:
                    differences[res][field][observable] = None

    # Compute RMSDs
    rmsd = {}
    for field, obs_dict in rmsd_data.items():
        rmsd[field] = {}
        for observable, diffs in obs_dict.items():
            diffs = np.array(diffs)
            rmsd[field][observable] = float(np.sqrt(np.mean(diffs**2))) if diffs.size > 0 else None

    return {"differences": differences, "rmsd": rmsd}


import numpy as np

def compute_relaxation_rate_differences(dict_a, dict_b):
    """
    Compute differences in R1=1/T1, R2=1/T2, and hetNOE between two nested dictionaries,
    and calculate per-field RMSDs across residues.

    Parameters
    ----------
    dict_a, dict_b : dict
        Nested dictionaries of the form:
        {
            residue: {
                field: {
                    observable: {"value": float}
                }
            }
        }

    Returns
    -------
    results : dict
        {
            "differences": {
                residue: {
                    field: {
                        R1: diff (float or None),
                        R2: diff (float or None),
                        hetNOE: diff (float or None)
                    }
                }
            },
            "rmsd": {
                field: {
                    R1: rmsd_value (float or None),
                    R2: rmsd_value (float or None),
                    hetNOE: rmsd_value (float or None)
                }
            }
        }
        Differences are dict_a - dict_b.
    """
    differences = {}
    rmsd_data = {}

    residues = set(dict_a.keys()) | set(dict_b.keys())

    for res in residues:
        differences[res] = {}
        fields = set(dict_a.get(res, {}).keys()) | set(dict_b.get(res, {}).keys())
        for field in fields:
            differences[res][field] = {}
            # R1 = 1/T1
            for obs, new_obs in [("T1", "R1"), ("T2", "R2")]:
                val_a = dict_a.get(res, {}).get(field, {}).get(obs, {}).get("value")
                val_b = dict_b.get(res, {}).get(field, {}).get(obs, {}).get("value")
                if val_a and val_b:  # avoid division by zero or None
                    r_a, r_b = 1.0 / val_a, 1.0 / val_b
                    diff = r_a - r_b
                    differences[res][field][new_obs] = diff
                    rmsd_data.setdefault(field, {}).setdefault(new_obs, []).append(diff)
                else:
                    differences[res][field][new_obs] = None

            # hetNOE (no inversion)
            val_a = dict_a.get(res, {}).get(field, {}).get("hetNOE", {}).get("value")
            val_b = dict_b.get(res, {}).get(field, {}).get("hetNOE", {}).get("value")
            if val_a is not None and val_b is not None:
                diff = val_a - val_b
                differences[res][field]["hetNOE"] = diff
                rmsd_data.setdefault(field, {}).setdefault("hetNOE", []).append(diff)
            else:
                differences[res][field]["hetNOE"] = None

    # Compute RMSDs
    rmsd = {}
    for field, obs_dict in rmsd_data.items():
        rmsd[field] = {}
        for observable, diffs in obs_dict.items():
            diffs = np.array(diffs)
            rmsd[field][observable] = float(np.sqrt(np.mean(diffs**2))) if diffs.size > 0 else None

    return {"differences": differences, "rmsd": rmsd}

import math
import numpy as np

def compute_chemical_shift_differences(dict1, dict2):
    """
    Compute chemical shift differences and RMSDs between two datasets.
    
    Parameters
    ----------
    dict1, dict2 : dict
        Dictionaries of the form:
        {
            residue_number: {
                nucleus: value,
                ...
            },
            ...
        }
    
    Returns
    -------
    results : dict
        {
            "differences": {
                residue_number: {
                    nucleus: difference (dict1 - dict2),
                    ...
                },
                ...
            },
            "rmsd": {
                nucleus: RMSD across all residues
            }
        }
    """
    differences = {}
    diff_by_nucleus = {}

    # Iterate over residues present in both
    for res in set(dict1.keys()).intersection(dict2.keys()):
        differences[res] = {}
        for nucleus in set(dict1[res].keys()).intersection(dict2[res].keys()):
            val1 = dict1[res][nucleus]
            val2 = dict2[res][nucleus]
            diff = val1 - val2
            differences[res][nucleus] = diff
            diff_by_nucleus.setdefault(nucleus, []).append(diff)

    # Compute RMSD per nucleus
    rmsd = {}
    for nucleus, diffs in diff_by_nucleus.items():
        if diffs:
            rmsd[nucleus] = math.sqrt(np.mean(np.square(diffs)))
        else:
            rmsd[nucleus] = None

    return {
        "differences": differences,
        "rmsd": rmsd
    }

def parse_trj_info(ID,systems):
    for system in systems:
        if system['ID'] != ID:
            continue
        data_folder = '../../Data/Simulations/' + system['path']
        trj_path = data_folder + system['TRJ'][0][0]
        """Extract replica name and force field from a trajectory file path."""
        fname = os.path.basename(trj_path)  # e.g. replica_03_AMBER03WS_md_1500ns.xtc
        parts = fname.split("_")
        replica = parts[0] + "_" + parts[1]   # "replica_03"
        forcefield = parts[2]                 # "AMBER03WS"
        return replica, forcefield

import os
import subprocess
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import gc

def process_pair(IDs, systems):
    """
    Process and analyze a pair of molecular dynamics trajectories.

    This function locates trajectory files for two specified simulation systems,
    performs preprocessing with GROMACS (`trjconv`), loads the processed trajectories
    into `Ensemble` objects, and computes similarity scores and feature differences
    between the two ensembles.

    Parameters
    ----------
    IDs : list of str or int
        Identifiers of the two systems to analyze. Only systems with matching IDs
        from the `systems` input are processed.
    systems : list of dict
        A list of system descriptors, where each dictionary must contain:
          - 'ID' : str or int
              System identifier.
          - 'path' : str
              Relative path to the simulation data folder.
          - 'TRJ' : list
              Nested list with at least one entry, where the first item is the
              trajectory filename (XTC format).

    Returns
    -------
    dict
        Dictionary containing analysis metrics:
        - "ada_score" : float
            Jensen–Shannon divergence (ADA features).
        - "ata_score" : float
            Jensen–Shannon divergence (ATA features).
        - "rama_score" : float
            Jensen–Shannon divergence (Ramachandran features).
        - "rg_diff" : float
            Difference in mean radius of gyration between the two systems.

    Notes
    -----
    - This function assumes GROMACS is installed and accessible at
      `/usr/local/gromacs/bin/GMXRC`.
    - Processed trajectories (`nojump_traj.xtc`, `skipped_traj.xtc`) are written
      in the same folder as the original trajectory file.
    - Requires custom classes `Ensemble` and `EnsembleAnalysis` as well as
      scoring functions `score_adaJSD`, `score_ataJSD`, and `score_ramaJSD`.

    """
    ensembles = []

    file_paths = []   # now local, populated from systems
    for system in systems:
        ID = system['ID']
        if ID in IDs:
            data_folder = '../../Data/Simulations/' + system['path']
            trj = data_folder + system['TRJ'][0][0]
            gro = data_folder + 'protein_centered.gro'
            file_paths.append((ID, trj, gro))
    
    for ID, xtc_path, gro_path in file_paths:
        fname = os.path.splitext(os.path.basename(xtc_path))[0]
        code = ID
        folder_path = os.path.dirname(os.path.abspath(xtc_path))
    
        nojump_traj = folder_path + '/nojump_traj.xtc'
        skipped_traj = folder_path + '/skipped_traj.xtc'
        skip_value = 10

        if not os.path.exists(skipped_traj):
            cmd = f"""
            source /usr/local/gromacs/bin/GMXRC && \
            echo System | gmx trjconv -f {xtc_path} -s {gro_path} -skip {skip_value} -o {nojump_traj} -pbc nojump  && \
            echo -e "System\nSystem" | gmx trjconv -f {nojump_traj} -s {gro_path} -o {skipped_traj} -center
            """
            subprocess.run(cmd, shell=True, executable="/bin/bash", check=True)
    
        ens = Ensemble(code=code, data_path=skipped_traj, top_path=gro_path)
        ens.load_trajectory()
        ensembles.append(ens)

    analysis = EnsembleAnalysis(ensembles)
    analysis.load_trajectories()

    code_list = [e.code for e in analysis.ensembles]
    ens_1, ens_2 = analysis[code_list[0]], analysis[code_list[1]]

    print(f"Systems {code_list[0]} and {code_list[1]} analyzed")

    ens_1_rg = np.mean(ens_1.get_features("rg"))
    ens_2_rg = np.mean(ens_2.get_features("rg"))
    rg_diff = ens_1_rg - ens_2_rg
    
    ada_score, _ = score_adaJSD(ens_1, ens_2, bins="auto", return_bins=True)
    ata_score, _ = score_ataJSD(ens_1, ens_2, bins="auto", return_bins=True)
    rama_score, _ = score_ramaJSD(ens_1, ens_2, bins=5, return_bins=True)

    # cleanup
    del ens_1, ens_2, analysis
    gc.collect()

    return {
        "ada_score": ada_score,
        "ata_score": ata_score,
        "rama_score": rama_score,
        "rg_diff": rg_diff,
    }

def make_rmsd_scores(rmsd_data):
    # Step 1: find minimum RMSD per class
    min_rmsd_per_class = {}
    for sim, values in rmsd_data.items():
        for cls, rmsd in values.items():
            if rmsd is None:
                continue
            if cls not in min_rmsd_per_class or rmsd < min_rmsd_per_class[cls]:
                min_rmsd_per_class[cls] = rmsd

    # Step 2+3: compute CSScore for each simulation
    cs_scores = {}
    for sim, values in rmsd_data.items():
        normed = []
        for cls, rmsd in values.items():
            if rmsd is None or cls not in min_rmsd_per_class:
                continue
            normed.append(rmsd / min_rmsd_per_class[cls])
        cs_scores[sim] = np.mean(normed) if normed else None

    return cs_scores

# Results
#print("Minimum RMSD per class:", min_rmsd_per_class)
#print("CS Scores per simulation:")
#for sim, score in cs_scores.items():
#    print(f"  {sim}: {score:.3f}")

def calculate_backbone_correlation_distributions(gro_file, xtc_file, output_file, bins=50):

    outfile_png = output_file
    outfile_csv = output_file.replace(".png", ".csv")
    outfile_dist = output_file.replace(".png", "_dist.csv")

    u = mda.Universe(gro_file, xtc_file)
    CAatoms = u.select_atoms("name CA")

    num_atoms = len(CAatoms)
    num_vectors = num_atoms - 1
    vec = np.zeros((num_vectors, 3))
    avg_matrix = np.zeros((num_vectors, num_vectors))

    # Histogram setup
    bin_edges = np.linspace(-1, 1, bins + 1)
    histograms = np.zeros((num_vectors, num_vectors, bins), dtype=int)

    for ts in u.trajectory:
        # Build backbone vectors
        for i in range(num_vectors):
            vec[i] = CAatoms[i + 1].position - CAatoms[i].position
            vec[i] /= np.linalg.norm(vec[i])
        corr_matrix = np.dot(vec, vec.T)

        avg_matrix += corr_matrix

        # Update histograms
        for i in range(num_vectors):
            for j in range(num_vectors):
                val = corr_matrix[i, j]
                bin_idx = np.searchsorted(bin_edges, val, side="right") - 1
                if 0 <= bin_idx < bins:
                    histograms[i, j, bin_idx] += 1

    # Normalize average matrix
    avg_matrix /= len(u.trajectory)

    # Save average matrix
    np.savetxt(outfile_csv, avg_matrix, delimiter=",")

    # Save histograms (flattened table for easier use later)
    col_names = [f"pair_{i}_{j}" for i in range(num_vectors) for j in range(num_vectors)]
    df_hist = pd.DataFrame(
        histograms.reshape(num_vectors * num_vectors, bins).T,
        columns=col_names
    )
    df_hist["bin_left"] = bin_edges[:-1]
    df_hist["bin_right"] = bin_edges[1:]
    df_hist.to_csv(outfile_dist, index=False)

from scipy.spatial.distance import jensenshannon

def calculate_js_matrix_IDs(ID1, ID2, systems, output_csv, output_png=None):
    """
    Calculate the Jensen-Shannon divergence matrix between distributions
    stored in two *_dist.csv files (created by calculate_backbone_correlation_distributions).

    Parameters
    ----------
    dist_file1 : str
        Path to first *_dist.csv file
    dist_file2 : str
        Path to second *_dist.csv file
    output_csv : str
        Path to save JS divergence matrix as CSV
    output_png : str, optional
        Path to save heatmap as PNG (if given)

    Returns
    -------
    js_matrix : np.ndarray
        Square matrix of JS divergence values
    js_mean : float
        Average JS divergence across all pairs
    """

    for system in systems:
        ID = system['ID']
        if ID == ID1:
            data_folder = '../../Data/Simulations/' + system['path']
            dist_file1 = data_folder + 'backbone_correlation_distribution.csv'
        if ID == ID2:
            data_folder = '../../Data/Simulations/' + system['path']
            dist_file2 = data_folder + 'backbone_correlation_distribution.csv'
        else:
            continue
    
    # Load both distribution files
    df1 = pd.read_csv(dist_file1)
    df2 = pd.read_csv(dist_file2)

    # Ensure same bins
    if not np.allclose(df1["bin_left"], df2["bin_left"]):
        raise ValueError("Bin edges in the two files do not match.")
    
    # Drop bin edges
    df1 = df1.drop(columns=["bin_left", "bin_right"])
    df2 = df2.drop(columns=["bin_left", "bin_right"])

    # Ensure same columns
    if list(df1.columns) != list(df2.columns):
        raise ValueError("Pair columns do not match between files.")
    
    col_names = df1.columns
    num_pairs = len(col_names)
    n_vectors = int(np.sqrt(num_pairs))

    # Calculate JS divergence per pair
    js_values = []
    for col in col_names:
        p = df1[col].values.astype(float)
        q = df2[col].values.astype(float)

        # Normalize to probability distributions
        p = p / p.sum() if p.sum() > 0 else np.zeros_like(p)
        q = q / q.sum() if q.sum() > 0 else np.zeros_like(q)

        # JS divergence
        js = jensenshannon(p, q, base=2) ** 2  # scipy returns sqrt(JS), so square it
        js_values.append(js)

    # Reshape into square matrix
    js_matrix = np.array(js_values).reshape(n_vectors, n_vectors)

    # Compute average divergence
    js_mean = np.nanmean(js_matrix)

    # Save CSV
    np.savetxt(output_csv, js_matrix, delimiter=",")

    # Optional: plot heatmap
    if output_png is not None:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6,5))
        im = plt.imshow(js_matrix, cmap="viridis", origin="lower")
        cbar = plt.colorbar(im)
        cbar.ax.tick_params(labelsize=12)
        plt.xlabel("C$_{\\alpha}$ Pair Index", fontsize=14)
        plt.ylabel("C$_{\\alpha}$ Pair Index", fontsize=14)
        plt.title("Jensen-Shannon Divergence Matrix", fontsize=14)
        plt.tight_layout()
        plt.savefig(output_png, dpi=600)
        plt.close()

    return js_matrix, js_mean

from scipy.spatial.distance import jensenshannon
def calculate_js_matrix(dist_file1, dist_file2, output_csv, output_png=None):
    """
    Calculate the Jensen-Shannon divergence matrix between distributions
    stored in two *_dist.csv files (created by calculate_backbone_correlation_distributions).

    Parameters
    ----------
    dist_file1 : str
        Path to first *_dist.csv file
    dist_file2 : str
        Path to second *_dist.csv file
    output_csv : str
        Path to save JS divergence matrix as CSV
    output_png : str, optional
        Path to save heatmap as PNG (if given)

    Returns
    -------
    js_matrix : np.ndarray
        Square matrix of JS divergence values
    js_mean : float
        Average JS divergence across all pairs
    """
    # Load both distribution files
    df1 = pd.read_csv(dist_file1)
    df2 = pd.read_csv(dist_file2)

    # Ensure same bins
    if not np.allclose(df1["bin_left"], df2["bin_left"]):
        raise ValueError("Bin edges in the two files do not match.")
    
    # Drop bin edges
    df1 = df1.drop(columns=["bin_left", "bin_right"])
    df2 = df2.drop(columns=["bin_left", "bin_right"])

    # Ensure same columns
    if list(df1.columns) != list(df2.columns):
        raise ValueError("Pair columns do not match between files.")
    
    col_names = df1.columns
    num_pairs = len(col_names)
    n_vectors = int(np.sqrt(num_pairs))

    # Calculate JS divergence per pair
    js_values = []
    for col in col_names:
        p = df1[col].values.astype(float)
        q = df2[col].values.astype(float)

        # Normalize to probability distributions
        p = p / p.sum() if p.sum() > 0 else np.zeros_like(p)
        q = q / q.sum() if q.sum() > 0 else np.zeros_like(q)

        # JS divergence
        js = jensenshannon(p, q, base=2) ** 2  # scipy returns sqrt(JS), so square it
        js_values.append(js)

    # Reshape into square matrix
    js_matrix = np.array(js_values).reshape(n_vectors, n_vectors)

    # Compute average divergence
    js_mean = np.nanmean(js_matrix)

    # Save CSV
    np.savetxt(output_csv, js_matrix, delimiter=",")

    # Optional: plot heatmap
    if output_png is not None:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6,5))
        im = plt.imshow(js_matrix, cmap="viridis", origin="lower")
        cbar = plt.colorbar(im)
        cbar.ax.tick_params(labelsize=12)
        plt.xlabel("C$_{\\alpha}$ Pair Index", fontsize=14)
        plt.ylabel("C$_{\\alpha}$ Pair Index", fontsize=14)
        plt.title("Jensen-Shannon Divergence Matrix", fontsize=14)
        plt.tight_layout()
        plt.savefig(output_png, dpi=600)
        plt.close()

    return js_matrix, js_mean

import json
import pandas as pd

def load_and_flatten(input_file, output_csv=None):
    """
    Load list of dicts from a JSON file and flatten into a pandas DataFrame.

    Parameters
    ----------
    input_file : str
        Path to JSON file containing the list of dicts
    output_csv : str, optional
        If given, save the resulting DataFrame to CSV

    Returns
    -------
    df : pandas.DataFrame
        Flattened DataFrame
    """
    # Load JSON data
    with open(input_file, "r") as f:
        data = json.load(f)

    # Flatten entries
    flat_rows = []
    for d in data:
        row = {
            "ID": d["ID"],
            "Replica": d["simulation"][0],
            "Forcefield": d["simulation"][1],
            "Rank": d["Rank"],
            "backbone_comparison": d["backbone_comparison"],
        }

        # Expand ensemble_comparison if it's a dict
        if isinstance(d["ensemble_comparison"], dict):
            row.update(d["ensemble_comparison"])
        else:
            row["ada_score"] = None
            row["ata_score"] = None
            row["rama_score"] = None
            row["rg_diff"] = None

        flat_rows.append(row)

    # Build DataFrame
    df = pd.DataFrame(flat_rows)

    # Sort by Rank
    df = df.sort_values("Rank").reset_index(drop=True)

    # Optional save
    if output_csv:
        df.to_csv(output_csv, index=False)

    return df


# Example usage
#if __name__ == "__main__":
#    df = load_and_flatten("input.json", "summary_table.csv")
#    print(df.to_string(index=False))


### Create pairs of FAIRDMD databank IDs for a set of simulations of a given protein

In [4]:
import itertools

#protein = 'ChiZ1-64'
#protein = 'KRS1-72'
protein = 'asyn'
#protein = 'icl2'

all_IDs = []
for system in systems:
    if protein in system['SYSTEM']:
        all_IDs.append(system['ID'])

print('Found IDs: ', all_IDs)

pairs = list(itertools.combinations(all_IDs, 2))

Found IDs:  [87, 93, 90, 71, 76, 88, 61, 65, 66, 73, 69, 57, 91, 98, 63, 77, 81, 74, 89, 86, 102, 52, 53, 92, 62]


In [14]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_system(system):
    ID = system['ID']
    output_file = f"../../Data/Simulations/{system['path']}backbone_correlation_distribution.csv"

    if os.path.isfile(output_file):
        return f"Skipping {ID}: output already exists."

    print(f"Calculating for {ID} at ../../Data/Simulations/{system['path']}")
    xtc_file = f"../../Data/Simulations/{system['path']}{system['TRJ'][0][0]}"
    gro_file = f"../../Data/Simulations/{system['path']}protein_centered.gro"

    calculate_backbone_correlation_distributions(gro_file, xtc_file, output_file)
    return f"Finished {ID}"

# ---- Main parallel loop ----
with ProcessPoolExecutor(max_workers=11) as executor:
    futures = [executor.submit(process_system, system) for system in systems if system['ID'] in all_IDs]

    for future in as_completed(futures):
        try:
            print(future.result())
        except Exception as e:
            print(f"Error: {e}")


Calculating for 87 at ../../Data/Simulations/f27/981/f27981c4bde82a195ed77bed88eb0cc45699e96f/5b0d5aeac77e7a6370f46d4e0514087397ed2e70/
Calculating for 93 at ../../Data/Simulations/9c5/976/9c59768006ed1c2b3062b5fa5a922779745a52af/3fbbd429fabc8ad767f23f73c66f8d822b3583c5/
Calculating for 90 at ../../Data/Simulations/881/4e0/8814e007dac99fea5b0cca9fbb77c0751fd495f2/546ed4699f283c499890ecda614ee11f95b129d9/Calculating for 71 at ../../Data/Simulations/e3b/7dd/e3b7dd11ed4af5ae33ba5576b0b9a6efc0c51bc4/d6aac8161e4daf70fe26ff690148d100d5f22886/

Calculating for 76 at ../../Data/Simulations/032/789/032789eb684106cf23f23aa69db12464c8953e9d/5ef698da2afc99ec6a97313d9d581b7d603620f8/
Calculating for 61 at ../../Data/Simulations/46e/de0/46ede08094301a656ae4035f2d38156c16d03da8/a55b23a027ea3f0b6580c382f2458699afcdd32c/Calculating for 88 at ../../Data/Simulations/91e/eab/91eeab5c65aa45651beca8c9ab24f39f9e380a18/58e6c9fb76bd48c95feaddd76bb6e7e345581243/Calculating for 65 at ../../Data/Simulations/03a/e

In [ ]:
for system in systems:
    ID = system['ID']
    if ID in all_IDs:
        output_file = '../../Data/Simulations/' + system['path'] + 'backbone_correlation_distribution.csv'
        if os.path.isfile(output_file):
            continue
        print('Calculating for ', ID, 'at ../../Data/Simulations/' + system['path'])
        xtc_file = '../../Data/Simulations/' + system['path'] +  system['TRJ'][0][0]
        gro_file = '../../Data/Simulations/' + system['path'] +  'protein_centered.gro'
        calculate_backbone_correlation_distributions(gro_file, xtc_file, output_file)

Calculating for  102 at ../../Data/Simulations/77e/6be/77e6be15918d603f28a24b5a4533bbad3aedcda3/b081fd2d1208b2d5fcb48e2f9cee5d5a65938e34/


In [5]:
rmsd_data = {}
sr_rmsd_data = {}

for system in systems:
    ID = system['ID']
    if ID in all_IDs:
        data_folder = '../../Data/Simulations/' + system['path']
        spin_relaxation_file_rmsd = data_folder + 'spin_relaxation_rmsd.yaml'
        chemical_shift_file_rmsd = data_folder + 'chemical_shift_rmsd.yaml'
        with open(spin_relaxation_file_rmsd, "r") as f:
            spin_relaxation_rmsd = yaml.safe_load(f)
        with open(chemical_shift_file_rmsd, "r") as f:
            chemical_shift_rmsd = yaml.safe_load(f)
        # Step 1: extract the top-level RMSD block (non-integer keys)
        rmsd_block = {k: v for k, v in chemical_shift_rmsd.items() if k != "differences" and not isinstance(k, int)}
        # Step 2: wrap into the rmsd_data format
        rmsd_data[ID] = rmsd_block
        # Step 1: extract the top-level RMSD block (non-integer keys)
        sr_rmsd_block = {k: v for k, v in spin_relaxation_rmsd.items() if k != "differences" and not isinstance(k, int)}
        # Step 2: wrap into the rmsd_data format
        sr_rmsd_data[ID] = sr_rmsd_block

In [6]:
cs_scores = make_rmsd_scores(rmsd_data)
sr_scores = make_rmsd_scores(sr_rmsd_data)

In [7]:
sr_scores

{87: 1.6111619799692238,
 93: 1.8028450350302085,
 90: 2.637743838347744,
 71: 2.3420544305881226,
 76: 2.1832344711286322,
 88: 1.8819320263525767,
 61: 2.3334422018102985,
 65: 1.558019709988157,
 66: 1.7132915933438948,
 73: 1.4489485318028799,
 69: 1.6572802160327977,
 57: 4.95752679102567,
 91: 1.6362479139874961,
 98: 1.5788242644891213,
 63: 1.5332786087818515,
 77: 1.3581089039258274,
 81: 1.554595141373391,
 74: 2.273024698132891,
 89: 1.666251866429007,
 86: 2.0737118299049615,
 102: 1.2315042864857075,
 52: 1.5019013597234976,
 53: 1.7531080743726548,
 92: 1.3275382879642443,
 62: 2.4305484381816354}

In [8]:
cs_scores

{87: 1.9642092806712408,
 93: 1.592426235712962,
 90: 1.7141604794706957,
 71: 2.677065437853595,
 76: 2.279815024948439,
 88: 1.2470321130906041,
 61: 2.866186016483052,
 65: 1.6674427261755216,
 66: 1.849481817091694,
 73: 1.4431506400087764,
 69: 1.6591276129440211,
 57: 2.4756194430227336,
 91: 1.4518359224199293,
 98: 1.547941274553385,
 63: 1.5918037207745592,
 77: 1.6522476933309556,
 81: 1.8248004750951494,
 74: 2.21480359489271,
 89: 1.621641940492212,
 86: 1.660201512847589,
 102: 1.517562609507532,
 52: 1.6454395220008202,
 53: 1.8607570498410007,
 92: 1.524967621236403,
 62: 2.91205961033826}

In [9]:
from scipy.stats import spearmanr, kendalltau, rankdata
import matplotlib.pyplot as plt

dict1 = cs_scores
dict2 = sr_scores

# Common IDs
common_ids = list(dict1.keys() & dict2.keys())

# Extract values in matching order
vals1 = [dict1[i] for i in common_ids]
vals2 = [dict2[i] for i in common_ids]

# Compute rank orders (ascending, so smallest value = rank 1)
ranks1 = rankdata(vals1, method="ordinal")
ranks2 = rankdata(vals2, method="ordinal")

print("\nRanking in List 2 spin relaxation times (ID -> Value -> Rank):")
first_sr = None
sr_ranking = []
for i, v, r in sorted(zip(common_ids, vals2, ranks2), key=lambda x: x[2]):
    if first_sr is None:
        first_sr = i
        ensemble_comparison = 0
        backbone_comparison = 0
    else:
        IDs = [first_sr,i]
        #print(i)
        ensemble_comparison = process_pair(IDs, systems)
        output_csv = "js_matrix_sr" + str(r) + ".csv"
        output_png = "js_matrix_sr" + str(r) + ".png"
        js_matrix, backbone_comparison = calculate_js_matrix_IDs(first_sr, i, systems, output_csv, output_png)

    #print(ensemble_comparison)
    replica, ff = parse_trj_info(i,systems)
    print(f"ID {i,replica,ff}: Value {v:.4f}, Rank {r}, {ensemble_comparison}, 'Backbone comparison', {backbone_comparison}")
    sr_ranking.append(
        {'ID':i, 
         'value': v,
         'simulation': (replica,ff), 
         'Rank': r, 
         'ensemble_comparison': ensemble_comparison,
         'backbone_comparison': backbone_comparison
        }
    )


# Print rank orders with values
print("Ranking in List 1 chemical shifts (ID -> Value -> Rank):")
first_cs = None
cs_ranking = []
for i, v, r in sorted(zip(common_ids, vals1, ranks1), key=lambda x: x[2]):
    if first_cs is None:
        first_cs = i
        ensemble_comparison = 0
        backbone_comparison = 0
    else:
        IDs = [first_cs,i]
        #print(i)
        ensemble_comparison = process_pair(IDs, systems)
        output_csv = "js_matrix_cs" + str(r) + ".csv"
        output_png = "js_matrix_cs" + str(r) + ".png"
        js_matrix, backbone_comparison = calculate_js_matrix_IDs(first_cs, i, systems, output_csv, output_png)
    
    #print(ensemble_comparison)
    replica, ff = parse_trj_info(i,systems)
    print(f"ID {i,replica,ff}: Value {v:.4f}, Rank {r}, {ensemble_comparison}, 'Backbone comparison', {backbone_comparison}")
    cs_ranking.append(
        {'ID':i, 
         'value': v,
         'simulation': (replica,ff), 
         'Rank': r, 
         'ensemble_comparison': ensemble_comparison,
         'backbone_comparison': backbone_comparison
        }
    )



# Compute correlations
spearman_corr, _ = spearmanr(vals1, vals2)
kendall_corr, _ = kendalltau(vals1, vals2)

print("\nSpearman rank correlation:", spearman_corr)
print("Kendall's tau correlation:", kendall_corr)

# Plot rankings
plt.figure(figsize=(8,6))
plt.scatter(ranks1, ranks2, color="blue")

# Add diagonal reference line
plt.plot([1, len(common_ids)], [1, len(common_ids)], "r--")

# Label points with ID
for i, (x, y) in enumerate(zip(ranks1, ranks2)):
    plt.text(x+0.1, y+0.1, str(common_ids[i]), fontsize=8)

plt.xlabel("Ranking in List 1 (smallest = 1)")
plt.ylabel("Ranking in List 2 (smallest = 1)")
plt.title("Comparison of Rankings between List 1 and List 2")
plt.grid(True)
plt.show()

Loading trajectory for 102...



Ranking in List 2 spin relaxation times (ID -> Value -> Rank):
ID (102, 'replica_01', 'AMBER03WS'): Value 1.2315, Rank 1, 0, 'Backbone comparison', 0


Loading trajectory for 92...
Loading trajectory for 102...
Loading trajectory for 92...


Systems 102 and 92 analyzed


FileNotFoundError: [Errno 2] No such file or directory: '../../Data/Simulations/77e/6be/77e6be15918d603f28a24b5a4533bbad3aedcda3/b081fd2d1208b2d5fcb48e2f9cee5d5a65938e34/backbone_correlation_distribution.csv'

In [10]:
# Save to file
sr_ranking_file = "sr_ranking" + protein +".json"
with open(sr_ranking_file, "w") as f:
    json.dump(sr_ranking, f, indent=2, default=lambda x: float(x) if isinstance(x, (np.integer, np.floating)) else str(x))

cs_ranking_file = "cs_ranking" + protein +".json"
with open(cs_ranking_file, "w") as f:
    json.dump(cs_ranking, f, indent=2, default=lambda x: float(x) if isinstance(x, (np.integer, np.floating)) else str(x))



In [11]:
load_and_flatten(sr_ranking_file, output_csv=None)

,ID,Replica,Forcefield,Rank,backbone_comparison,ada_score,ata_score,rama_score,rg_diff
0,92,replica_05,AMBER03WS,1.0,0.000000,NaN,NaN,NaN,NaN
1,77,replica_04,AMBER03WS,2.0,0.019829,0.126350,0.052044,0.041670,-0.255408
2,73,replica_05,AMBER99SB-DISP,3.0,0.023387,0.112694,0.114226,0.117995,0.075787
3,52,replica_03,AMBER99SBWS,4.0,0.014943,0.171157,0.108970,0.116673,2.731402
4,81,replica_03,AMBER03WS,5.0,0.029648,0.133781,0.081962,0.057954,-0.259605
5,63,replica_02,AMBER99SBWS,6.0,0.011996,0.082835,0.093911,0.113079,0.145076
6,65,replica_04,AMBER99SB-DISP,7.0,0.021843,0.106550,0.131964,0.132087,-0.110866
7,98,replica_01,AMBER99SBWS,8.0,0.011545,0.061969,0.090322,0.107163,0.021662
8,87,replica_02,AMBER03WS,9.0,0.021965,0.099541,0.108603,0.089516,-0.276942
9,91,replica_05,DESAMBER,10.0,0.035243,0.177290,0.121107,0.130602,-0.399593


In [12]:
load_and_flatten(cs_ranking_file, output_csv=None)

,ID,Replica,Forcefield,Rank,backbone_comparison,ada_score,ata_score,rama_score,rg_diff
0,88,replica_05,CHARMM36M,1.0,0.000000,NaN,NaN,NaN,NaN
1,73,replica_05,AMBER99SB-DISP,2.0,0.049358,0.235629,0.127989,0.118171,-0.585497
2,91,replica_05,DESAMBER,3.0,0.062098,0.232732,0.141046,0.139747,-0.110116
3,92,replica_05,AMBER03WS,4.0,0.041807,0.197231,0.120118,0.125731,-0.509710
4,98,replica_01,AMBER99SBWS,5.0,0.040792,0.194018,0.151718,0.171080,-0.531372
5,63,replica_02,AMBER99SBWS,6.0,0.041382,0.212912,0.173079,0.182989,-0.654785
6,93,replica_04,DESAMBER,7.0,0.051497,0.205694,0.162422,0.153745,0.309161
7,89,replica_03,AMBER99SB-DISP,8.0,0.053313,0.221346,0.181331,0.163580,-0.325441
8,52,replica_03,AMBER99SBWS,9.0,0.043346,0.323188,0.182001,0.189597,-3.241112
9,77,replica_04,AMBER03WS,10.0,0.049572,0.207909,0.132458,0.135993,-0.254302


In [26]:
data = sr_ranking
import pandas as pd

# Flatten entries
flat_rows = []
for d in data:
    row = {
        "ID": d["ID"],
        "Replica": d["simulation"][0],
        "Forcefield": d["simulation"][1],
        "Rank": d["Rank"],
        "Value": d["value"],
        "backbone_comparison": d["backbone_comparison"],
    }

    # Expand ensemble_comparison if it's a dict
    if isinstance(d["ensemble_comparison"], dict):
        row.update(d["ensemble_comparison"])
    else:
        row["ada_score"] = None
        row["ata_score"] = None
        row["rama_score"] = None
        row["rg_diff"] = None

    flat_rows.append(row)

# Make DataFrame
df = pd.DataFrame(flat_rows)

# Sort by Rank (optional)
df = df.sort_values("Rank")

# Pretty print
print(df.to_string(index=False))

# Save nicely if you want
df.to_csv("summary_table.csv", index=False)


 ID    Replica     Forcefield  Rank    Value  backbone_comparison  ada_score  ata_score  rama_score   rg_diff
 41 replica_04 AMBER99SB-DISP     1 1.320864             0.000000        NaN        NaN         NaN       NaN
 24 replica_01 AMBER99SB-DISP     2 1.356818             0.008548   0.072350   0.073066    0.041121  0.094428
 22 replica_05 AMBER99SB-DISP     3 1.370612             0.011823   0.088600   0.073018    0.044101  0.102528
 43 replica_02       DESAMBER     4 1.380707             0.032638   0.127577   0.134438    0.099024  0.387424
 12 replica_05      CHARMM36M     5 1.396430             0.072507   0.250961   0.185576    0.127174 -0.779634
 28 replica_01      CHARMM36M     6 1.398777             0.043672   0.233931   0.147362    0.122909 -0.921412
  7 replica_03      CHARMM36M     7 1.427920             0.042635   0.235422   0.119901    0.099898 -0.868865
 33 replica_04       DESAMBER     8 1.559637             0.013593   0.050335   0.072009    0.069033 -0.122194
 16 replic